# Testing Combined H5 Processing

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm
import h5py
import logging

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Single Event Testing

In [3]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0)

In [4]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_mini_pilot/ttbar/v7/runs/0/edm4hep.root"
event2 = EDM4hepEvent(edm_input_file, event_index=0)

In [9]:
print(f"""
Number of calo hits: {len(event.get_calo_hits_df())}
Number of particles: {len(event.get_particles_df())}
Number of tracker hits: {len(event.get_tracker_hits_df())}
""")


Number of calo hits: 1243953
Number of particles: 880327
Number of tracker hits: 240622



In [10]:
print(f"""
Number of calo hits: {len(event2.get_calo_hits_df())}
Number of particles: {len(event2.get_particles_df())}
Number of tracker hits: {len(event2.get_tracker_hits_df())}
""")


Number of calo hits: 1120404
Number of particles: 190319
Number of tracker hits: 217781



In [11]:
event2.get_calo_hits_df()

,cellID,energy,x,y,z,contribution_begin,contribution_end,detector,r,R,phi,theta,eta
0,152559394436048912,3.538352e-04,1190.580322,437.952515,2759.100098,0,9,ECalBarrelCollection,1268.575562,3036.760986,0.352486,0.430956,1.519249
1,4503814384687120,5.628845e-05,-274.604279,-1329.300415,81.599998,9,10,ECalBarrelCollection,1357.367676,1359.818237,-1.774509,1.510752,0.060080
2,1970496644814864,2.591849e-05,-333.317444,-1337.777222,35.700001,10,11,ECalBarrelCollection,1378.676270,1379.138428,-1.814982,1.544908,0.025892
3,5348204955310096,3.629815e-04,-325.826416,-1346.346069,96.900002,11,18,ECalBarrelCollection,1385.211426,1388.596558,-1.808239,1.500957,0.069896
4,64176406367602704,8.606262e-05,601.779175,-1106.322876,1162.800049,18,19,ECalBarrelCollection,1259.400024,1714.115601,-1.072613,0.825258,0.826083
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1120399,5629340626780692,2.156135e-09,1199.873169,342.497345,-3698.500000,5346121,5346123,HCalEndcapCollection,1247.798096,3903.319336,0.278050,2.816205,-1.807007
1120400,1407215976088084,5.042664e-04,1112.082520,-98.856026,-3647.500000,5346123,5346129,HCalEndcapCollection,1116.467651,3814.545410,-0.088660,2.844557,-1.899660
1120401,844300382437908,2.557875e-06,864.988586,-110.881462,-3698.500000,5346129,5346132,HCalEndcapCollection,872.066467,3799.921387,-0.127493,2.910033,-2.151582
1120402,5066390673457684,1.597710e-08,1188.167725,283.650238,-3851.500000,5346132,5346133,HCalEndcapCollection,1221.556396,4040.575684,0.234343,2.834465,-1.865733


In [12]:
event2.get_calo_contributions_df()

,PDG,energy,time,step_x,step_y,step_z,particle_id,cellID,x,y,z,detector
0,0,1.297844e-05,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
1,0,8.172441e-06,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
2,0,4.534445e-06,2562.023438,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
3,0,1.256829e-05,2562.023682,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
4,0,1.282355e-05,2562.023926,0.0,0.0,0.0,75414,0,1190.580322,437.952515,2759.100098,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...,...,...
5346129,0,9.253779e-07,187.780411,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346130,0,1.031067e-06,188.810455,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346131,0,6.014303e-07,189.794922,0.0,0.0,0.0,1036,201624,864.988586,-110.881462,-3698.500000,HCalEndcapCollection
5346132,0,1.597710e-08,1742.137573,0.0,0.0,0.0,1036,201625,1188.167725,283.650238,-3851.500000,HCalEndcapCollection


## Combined multi-output conversion (single open per run)

This workflow opens each `edm4hep.root` once per run and produces multiple H5 outputs (truth/particles and reco/tracker_hits) in one pass, reusing preloaded DataFrames to avoid repeated IO.


In [2]:
from pathlib import Path
import logging

# Reuse postprocessing helpers
sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/scripts/postprocessing")
from convert_particles import build_particles_df_with_parents_and_vertex, write_particles_with_selection
from convert_digihits import process_event_for_digihits, write_digihits_with_selection
from utils.path_utils import get_run_paths, make_dir
from utils.track_utils import load_root_file

from convert_all import convert_all


In [3]:
# Configurable column selection for H5 outputs
particles_columns_keep = [
    "particle_id", "pdg_id", "mass", "energy", "charge",
    "vx", "vy", "vz", "time", "px", "py", "pz",
    "num_tracker_hits", "num_calo_hits", "vertex_primary", "parent_id",
]
digihits_columns_keep = [
    "x", "y", "z", "time", "particle_id",
    "cell_id", "detector", "event_id",
]


In [4]:
# Config for this combined test (aligns with your production YAMLs)
campaign = "full_pileup_pilot"
dataset = "ttbar"
version = "v2"

base_root = Path("/pscratch/sd/d/danieltm/ColliderML/simulation")
output_base_dir = Path("./h5_testing/v2")  # unified root like scripts

logging.basicConfig(
    level=logging.DEBUG,  # show DEBUG and above
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,  # override prior configs in this kernel
)

config = {
    "chunk_size": 100,
    "run_size": 128,
    "max_chunks": 1,
    "campaign": "full_pileup_pilot",
    "dataset": "ttbar",
    "version": "v2",
    "common": {
        "output_base_dir": base_root,
    },
    "objects": ["particles", "tracker_hits"],
    "particles_columns_keep": particles_columns_keep,
    "digihits_columns_keep": digihits_columns_keep,
    "min_tracker_hits": 1,
    "h5_output_dir": output_base_dir,
}


In [ ]:
convert_all(config)

2025-09-11 04:14:06,677 - DEBUG - convert_all - Starting conversion with config: campaign=full_pileup_pilot, dataset=ttbar, version=v2
2025-09-11 04:14:06,678 - DEBUG - convert_all - Input base directory: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,679 - DEBUG - convert_all - Output base directory: h5_testing/v2
2025-09-11 04:14:06,679 - DEBUG - convert_all - Chunk size: 100, Run size: 128
2025-09-11 04:14:06,679 - DEBUG - convert_all - Objects to convert: ['particles', 'tracker_hits']
2025-09-11 04:14:06,679 - DEBUG - convert_all - Dataset base path: full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,680 - DEBUG - convert_all - Importing required modules completed
2025-09-11 04:14:06,689 - DEBUG - convert_all - Retrieved 109 run directories from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2
2025-09-11 04:14:06,694 - DEBUG - convert_all - Created output directories: particles=h5_testing/v2/full_pileup_pilot/ttbar/v

2025-09-11 04:14:06,779 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,780 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,790 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,799 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,801 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,808 - DEBUG - fsspec.local - open file: /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/particles.root
2025-09-11 04:14:06,850 - DEBUG - asyncio - Using selector: EpollSelec

## Test

In [5]:
import pandas as pd
import h5py

def load_all_particles(h5_path):
    with h5py.File(h5_path, 'r') as f:
        frames = []
        for ev_name in f['events'].keys():
            ev_id = int(ev_name.split('_')[1])
            arr = f['events'][ev_name]['particles'][:]
            df = pd.DataFrame(arr)
            df['event_id'] = ev_id
            frames.append(df)
        return pd.concat(frames, ignore_index=True)

def load_all_digihits(h5_path):
    with h5py.File(h5_path, 'r') as f:
        frames = []
        for ev_name in f['events'].keys():
            ev_id = int(ev_name.split('_')[1])
            arr = f['events'][ev_name]['measurements'][:]
            df = pd.DataFrame(arr)
            df['event_id'] = ev_id
            frames.append(df)
        return pd.concat(frames, ignore_index=True)

In [17]:
particles_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/truth/particles/full_pileup_pilot.ttbar.v2.truth.particles.events0-127.h5"
digihits_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/reco/tracker_hits/full_pileup_pilot.ttbar.v2.reco.tracker_hits.events0-127.h5"

In [18]:
particles_df = load_all_particles(particles_file)
digihits_df = load_all_digihits(digihits_file)

In [24]:
len(particles_df)/128

53639.6328125

In [29]:
particles_df[particles_df['event_id'] == 0]

,particle_id,pdg_id,mass,energy,charge,vx,vy,vz,time,px,py,pz,num_tracker_hits,num_calo_hits,vertex_primary,parent_id,event_id
0,81,2212,0.938270,2.088655,1.0,0.009780,-0.003074,-89.848335,-0.765510,-0.153240,0.357442,1.825070,12,67,1,10.0,0
1,83,-211,0.139570,0.411130,-1.0,0.009780,-0.003074,-89.848335,-0.765510,-0.349941,-0.123546,-0.108746,23,14,1,10.0,0
2,85,-211,0.139570,1.303886,-1.0,0.009780,-0.003074,-89.848335,-0.765510,-0.515224,1.013931,-0.622196,15,44,1,10.0,0
3,88,-211,0.139570,0.367787,-1.0,0.009780,-0.003074,-89.848335,-0.765510,0.120762,0.196509,-0.250176,10,26,1,10.0,0
4,89,211,0.139570,1.670073,1.0,0.009780,-0.003074,-89.848335,-0.765510,-1.257038,0.601183,-0.909999,13,99,1,10.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52097,879525,11,0.000511,0.001878,-1.0,-587.023193,36.333473,2192.566162,14.332582,-0.000897,-0.001288,-0.000895,3,0,1,879522.0,0
52098,879526,11,0.000511,0.000773,-1.0,242.825897,957.259094,2270.417969,10.273644,-0.000568,-0.000115,-0.000024,1,0,1,879522.0,0
52099,880214,2112,0.939565,0.939693,0.0,599.215210,1192.483765,2467.749512,12.798975,-0.002671,0.014943,-0.003215,1,0,1,880193.0,0
52100,880270,11,0.000511,0.001495,-1.0,44.648663,164.044464,238.886871,0.470167,0.001061,0.000171,0.000904,3,0,1,350.0,0


In [26]:
particles_df.columns

Index(['particle_id', 'pdg_id', 'mass', 'energy', 'charge', 'vx', 'vy', 'vz',
       'time', 'px', 'py', 'pz', 'num_tracker_hits', 'num_calo_hits',
       'vertex_primary', 'parent_id', 'event_id'],
      dtype='object')

In [27]:
len(digihits_df)/128

247057.84375

In [28]:
digihits_df

,x,y,z,time,particle_id,cell_id,detector,event_id
0,85.607491,3.613425,-1515.599976,35.864868,703493,67494740071,1,0
1,92.881134,-3.039804,-1515.599976,13.531095,67930,1023410279,1,0
2,66.319473,-7.858758,-1515.599976,9.396918,74845,2634023015,1,0
3,65.872543,-3.477791,-1515.599976,8655.912109,669642,1157628007,1,0
4,92.184578,-4.563632,-1515.599976,6.495801,375396,1526726759,1,0
...,...,...,...,...,...,...,...,...
31623399,417.815704,899.408813,3009.500000,52.765377,206765,1198296961374,5,99
31623400,413.691681,901.021179,3009.500000,13.214906,813341,1073742909790,5,99
31623401,372.737976,917.032959,3009.500000,16.807945,777212,17403208569182,5,99
31623402,372.626648,917.076477,3009.500000,16.808374,777213,17403208569182,5,99
